# xarray

> Utilities for handling Euclid FITS images as xarray Datasets, using dask for efficient computation and memory usage.

In [ ]:
# | default_exp euclid.xarray

In [ ]:
# | export

import json
from pathlib import Path
from warnings import catch_warnings, filterwarnings

import numpy as np
import pandas as pd
import fsspec
import numcodecs
import xarray as xr
import zarr
from astropy.io import fits
from astropy.io.fits import HDUList, ImageHDU, PrimaryHDU
from astropy.wcs import WCS
from fsspec.implementations.reference import LazyReferenceMapper
from kerchunk.codecs import AsciiTableCodec, VarArrCodec
from kerchunk.combine import MultiZarrToZarr
from tqdm import tqdm

from nicl.mask import fast_mask
from nicl.euclid.utilities import (
    get_dither_id_from_filename,
    get_filter_from_filename,
    get_obs_id_from_filename,
    get_tile_index_from_filename,
)

In [ ]:
# | hide
# # Additional imports used in the examples

from nicl.euclid.utilities import default_data_path, get_nisp_images_for_observation

In [ ]:
# | exporti


def _rename(refs, new_name):
    """Rename the zarray in a kerchunk json reference."""
    new_refs = {}
    for key in refs.keys():
        if "/" in key:
            parts = key[key.find("/") + 1 :]
            new_key = f"{new_name}/{parts}"
            new_refs[new_key] = refs[key]
    return new_refs

In [ ]:
# | exporti


def _get_ext_type(x):
    return x.split(".")[-1] if "." in x else None


def _get_ext_det(x):
    return ".".join(x.split(".")[:-1]) if "." in x else x

In [ ]:
# | exporti

# Code vendored from kerchunk.fits and adapted for speed with very large files

BITPIX2DTYPE = {
    8: "uint8",
    16: ">i2",
    32: ">i4",
    64: ">i8",
    -32: ">f4",
    -64: ">f8",
}  # always bigendian


def process_file(
    url,
    extension,
    storage_options=None,
    out=None,
    hdu=None,
    infile=None,
):
    """
    Create JSON references for a single FITS file as a zarr group

    Parameters
    ----------
    url: str
        Where the file is
    extension: str
        The extension name
    storage_options: dict
        How to load that file (passed to fsspec)
    out: dict-like or None
        This allows you to supply an fsspec.implementations.reference.LazyReferenceMapper
        to write out parquet as the references get filled, or some other dictionary-like class
        to customise how references get stored
    hdu: HDU or None
        An open FITS HDU to process
    infile: HDUList or None
        An open FITS file to process


    Returns
    -------
    dict of the references and the header
    """
    from astropy.io import fits

    storage_options = storage_options or {}
    out = out or {}
    g = zarr.open(out)

    with fsspec.open(url, mode="rb", **storage_options) as f:
        if hdu is None and infile is None:
            infile = fits.open(f, do_not_scale_image_data=True)
        if hdu is None:
            hdu = infile[extension]
        hdr = hdu.header
        hdr.__str__()  # causes fixing of invalid cards
        attrs = dict(hdr)
        kwargs = {}
        if hdu.is_image:
            # for images/cubes (i.e., ndarrays with simple type)
            nax = attrs["NAXIS"]
            shape = tuple(int(attrs[f"NAXIS{i}"]) for i in range(nax, 0, -1))
            dtype = BITPIX2DTYPE[attrs["BITPIX"]]
            length = np.dtype(dtype).itemsize
            for s in shape:
                length *= s

            if "BSCALE" in attrs or "BZERO" in attrs:
                kwargs["filters"] = [
                    numcodecs.FixedScaleOffset(
                        offset=float(attrs.get("BZERO", 0)),
                        scale=float(attrs.get("BSCALE", 1)),
                        astype=dtype,
                        dtype=float,
                    )
                ]
            else:
                kwargs["filters"] = None
        elif isinstance(hdu, fits.hdu.table.TableHDU):
            # ascii table
            spans = hdu.columns._spans
            outdtype = [
                [name, hdu.columns[name].format.recformat]
                for name in hdu.columns.names
            ]
            indtypes = [
                [name, f"S{i + 1}"] for name, i in zip(hdu.columns.names, spans)
            ]
            nrows = int(attrs["NAXIS2"])
            shape = (nrows,)
            kwargs["filters"] = [AsciiTableCodec(indtypes, outdtype)]
            dtype = [tuple(d) for d in outdtype]
            length = (sum(spans) + len(spans)) * nrows
        elif isinstance(hdu, fits.hdu.table.BinTableHDU):
            # binary table
            dtype = hdu.columns.dtype.newbyteorder(">")  # always big endian
            nrows = int(attrs["NAXIS2"])
            shape = (nrows,)
            # if hdu.fileinfo()["datSpan"] > length
            if any(_.format.startswith(("P", "Q")) for _ in hdu.columns):
                # contains var fields
                length = hdu.fileinfo()["datSpan"]
                dt2 = [
                    (name, "O")
                    if hdu.columns[name].format.startswith(("P", "Q"))
                    else (name, str(dtype[name].base))
                    + ((dtype[name].shape,) if dtype[name].shape else ())
                    for name in dtype.names
                ]
                types = {
                    name: hdu.columns[name].format[1]
                    for name in dtype.names
                    if hdu.columns[name].format.startswith(("P", "Q"))
                }
                kwargs["object_codec"] = VarArrCodec(
                    str(dtype), str(dt2), nrows, types
                )
                dtype = dt2
            else:
                length = dtype.itemsize * nrows
                kwargs["filters"] = None
        else:
            print(f"Unknown extension type: {hdu}")
        # one chunk for whole thing.
        # TODO: we could sub-chunk on biggest dimension
        name = hdu.name or str(extension)
        arr = g.empty(
            name, dtype=dtype, shape=shape, chunks=shape, compression=None, **kwargs
        )
        arr.attrs.update(
            {
                k: str(v) if not isinstance(v, (int, float, str)) else v
                for k, v in attrs.items()
                if k != "COMMENT"
            }
        )
        arr.attrs["_ARRAY_DIMENSIONS"] = ["z", "y", "x"][-len(shape) :]
        loc = hdu.fileinfo()["datLoc"]
        parts = ".".join(["0"] * len(shape))
        out[f"{name}/{parts}"] = [url, loc, length]
    if isinstance(out, LazyReferenceMapper):
        out.flush()
    return out, attrs


In [ ]:
# | export


def create_zarr_ref_from_fits(fns, out_path=None, progress=False):
    """Create a zarr reference from a set of Euclid fits files.

    This uses kerchunk to build a zarr reference file for accessing data directly from fits files.
    The resulting reference can then be opened as a xarray Dataset with `open_zarr_ref_as_dataset`.
    """
    if isinstance(fns, str):
        fns = [fns]
    else:
        fns = [str(fn) for fn in fns]
    fn = fns[0]
    if "MER" in str(fn):
        product_id_name = "tile_index"
        get_product_id = get_tile_index_from_filename
    else:
        product_id_name = "observation_id"
        get_product_id = get_obs_id_from_filename
    with fits.open(fn) as hdul:
        exts = list(h.name for h in hdul if h.size > 0)
    ref_files = []
    product_ids = []
    dither_ids = []
    filters = []
    wcs_list = []
    zp_list = []
    pbar = tqdm(fns) if progress else fns
    for fn in pbar:
        hdul = fits.open(fn, do_not_scale_image_data=True)
        product_id = get_product_id(fn)
        product_ids.append(product_id)
        dither_id = get_dither_id_from_filename(fn)
        if dither_id is not None:
            dither_ids.append(dither_id)
        filter = get_filter_from_filename(fn)
        filters.append(filter)
        if progress:
            pbar.set_description(
                f"{fn.split('/')[-1]}, {product_id}, {dither_id}, {filter}"
            )
        ref_exts = []
        coords = []
        coord_name = "extension"
        for ext in exts:
            hdu = hdul[ext]
            ext_type = _get_ext_type(ext)
            out, hdr = process_file(fn, hdu=hdu, extension=ext)
            if ext_type is not None:
                out = _rename(out, ext_type)
                coord_name = "detector"
            ref_exts.append(out)
            coord = _get_ext_det(ext)
            coords.append(coord)
            wcs = {
                k: hdr[k]
                for k in (
                    "CTYPE1",
                    "CTYPE2",
                    "CD1_1",
                    "CD1_2",
                    "CD2_1",
                    "CD2_2",
                    "CRPIX1",
                    "CRPIX2",
                    "CRVAL1",
                    "CRVAL2",
                    "NAXIS1",
                    "NAXIS2",
                )
            }
            wcs[coord_name] = coord
            if dither_id is not None:
                wcs["dither"] = dither_id
            if dither_id is not None:
                wcs[product_id_name] = product_id
            wcs_list.append(wcs)
            zp = {"ZP": hdr.get("ZPAB")}
            zp[coord_name] = coord
            if dither_id is not None:
                zp["filter"] = filter
            if dither_id is not None:
                zp["dither"] = dither_id
            if dither_id is not None:
                zp[product_id_name] = product_id
            zp_list.append(zp)
        mzz = MultiZarrToZarr(
            ref_exts,
            coo_map={coord_name: coords},
            concat_dims=[coord_name],
            identical_dims=["x", "y"],
        )
        ref_exts = mzz.translate()
        ref_files.append(ref_exts)
        hdul.close()
    coo_map = {}
    concat_dims = []
    if len(np.unique(product_ids)) > 0:
        coo_map[product_id_name] = product_ids
        concat_dims.append(product_id_name)
    if len(np.unique(dither_ids)) > 0:
        coo_map["dither"] = dither_ids
        concat_dims.append("dither")
    if len(np.unique(filters)) > 0:
        coo_map["filter"] = filters
        concat_dims.append("filter")
    mzz = MultiZarrToZarr(
        ref_files, coo_map=coo_map, concat_dims=concat_dims, identical_dims=["x", "y"]
    )
    with catch_warnings():
        filterwarnings(
            "ignore", "Concatenated coordinate .* contains less than expected"
        )
        ref = mzz.translate()
    wcs = pd.DataFrame(wcs_list)
    wcs = wcs.groupby([coord_name, product_id_name, "dither"]).first()
    wcs = wcs.to_xarray()
    zp = pd.DataFrame(zp_list)
    zp = zp.groupby([coord_name, product_id_name, "dither", "filter"]).first()
    zp = zp["ZP"].to_xarray()
    if out_path is not None:
        Path(out_path).mkdir(parents=True, exist_ok=True)
        with open(out_path / "ref.json", "w") as f:
            json.dump(ref, f)
        wcs.to_zarr(out_path / "wcs.zarr")
        zp.to_zarr(out_path / "zp.zarr")
    return (ref, wcs, zp)

In [ ]:
# | export


def open_zarr_ref_as_dataset(ref):
    """Open a zarr reference as a dask xarray Dataset."""
    ds = xr.open_dataset(
        "reference://",
        engine="zarr",
        chunks="auto",
        backend_kwargs={
            "storage_options": {
                "fo": ref,
            },
            "consolidated": False,
        },
    )
    return ds

In [ ]:
# | export


def open_fits_as_dataset(fns):
    """Create a zarr reference from a set of fits files and open as a dask xarray Dataset.

    If you plan to reopen the Dataset repeatedly, then is would be better to create the
    JSON reference once with `create_zarr_ref_from_fits`, save it, and then just open
    the reference as needed with `open_zarr_ref_as_dataset`.
    """
    ref = create_zarr_ref_from_fits(fns)
    ds = open_zarr_ref_as_dataset(ref)
    return ds

In [ ]:
# | export


def write_da_to_fits(
    da,  # the DataArray, with coordinates `x`, `y`, and `detector`
    fn,  # the filename to which to write the data
    da_wcs=None,  # an optional DataArray of WCS info for each `detector`
    overwrite=False,  # should any existing file be overwritten
):
    """Write a DataArray to a FITS file."""
    hdul = HDUList(PrimaryHDU())
    if "detector" in da.coords:
        for det in da["detector"].to_numpy():
            if da_wcs is not None:
                wcs = WCS(da_wcs.sel(detector=det).as_numpy().to_pandas())
                hdr = wcs.to_header()
            else:
                hdr = None
            hdul.append(ImageHDU(da.sel(detector=det), name=det, header=hdr))
    else:
        if da_wcs is not None:
            wcs = WCS(da_wcs.as_numpy().to_pandas())
            hdr = wcs.to_header()
        else:
            hdr = None
        hdul.append(ImageHDU(da, header=hdr))
    hdul.writeto(fn, overwrite=overwrite)
    return hdul

In [ ]:
# | export


def load_zarr_ref(path):
    with open(path / "ref.json") as f:
        ref = json.load(f)
    wcs = xr.open_zarr(path / "wcs.zarr")
    zp = xr.open_zarr(path / "zp.zarr")
    return ref, wcs, zp

In [ ]:
# | export


def _parse_wcs_dataset(wcs):
    """Parse a WCS Dataset into a DataArray of WCS objects."""
    wcs_objects = wcs.to_dataframe().apply(WCS, axis="columns")
    return wcs_objects


def load_wcs_from_zarr(fn):
    """Load a WCS Dataset from a zarr reference file."""
    wcs = xr.open_dataset(fn, engine="zarr")
    wcs = _parse_wcs_dataset(wcs)
    return wcs

In [ ]:
# | export


def create_all_zarr_refs(path, zarr_path, obs_id_glob="[23]*"):
    obs_ids = sorted([p.name for p in (path).glob(obs_id_glob)])
    for obs_id in obs_ids:
        fns = sorted([str(p) for p in (path / obs_id).glob("*.fits")])
        zarrfn = zarr_path / f"{obs_id}"
        if not zarrfn.exists():
            try:
                create_zarr_ref_from_fits(fns, out_path=zarrfn)
            except Exception as e:
                print(f"Error creating zarr ref for {obs_id}: {e}")
                continue

In [ ]:
# | export


def read_all_zarr_refs(zarr_path, obs_id_glob="[23]*"):
    ref_files = [str(p) for p in zarr_path.glob(f"{obs_id_glob}/ref.json")]
    datasets = [open_zarr_ref_as_dataset(ref) for ref in ref_files]
    ds = xr.concat(datasets, dim="observation_id")
    ds = ds.assign_coords(y=("y", ds.y.values), x=("x", ds.x.values))
    wcs_files = [str(p) for p in zarr_path.glob(f"{obs_id_glob}/wcs.zarr")]
    wcs = xr.open_mfdataset(wcs_files, engine="zarr", combine="nested", concat_dim="observation_id")
    zp_files = [str(p) for p in zarr_path.glob(f"{obs_id_glob}/zp.zarr")]
    zp = xr.open_mfdataset(zp_files, engine="zarr", combine="nested", concat_dim="observation_id")
    return ds, wcs, zp

In [ ]:
# | export


def xr_fast_mask(
    data,
    mask_params=None,  # mask parameters
    estimate_background=False,  # estimate the background from the image median
    max_repeat=10,  # maximum number of masking iterations
    relative_rms_tolerance=0.01,
    verbose=False,
):
    mask = xr.apply_ufunc(
        fast_mask,
        data,
        kwargs=dict(
            mask_params=mask_params,
            estimate_background=estimate_background,
            max_repeat=max_repeat,
            relative_rms_tolerance=relative_rms_tolerance,
            verbose=verbose,
            return_threshold=False,
        ),
        vectorize=True,
        dask="parallelized",
        input_core_dims=[["x", "y"]],
        output_core_dims=[["x", "y"]],
        output_dtypes=[bool],
    )
    return mask

## Example

In [ ]:
path = default_data_path("Q1_R1")
obsid = 2683
image_info = get_nisp_images_for_observation(obsid, path=path)
fns = image_info.filename

In [ ]:
ref, wcs, zp = create_zarr_ref_from_fits(fns, progress=True)

EUC_NIR_W-CAL-IMAGE_Y-2683-3_20240930T174928.130205Z.fits, 2683, 3, Y: 100%|██████████| 12/12 [00:18<00:00,  1.52s/it]


In [ ]:
wcs

<xarray.Dataset> Size: 6kB
Dimensions:         (detector: 16, observation_id: 1, dither: 4)
Coordinates:
  * detector        (detector) object 128B 'DET11' 'DET12' ... 'DET43' 'DET44'
  * observation_id  (observation_id) int64 8B 2683
  * dither          (dither) object 32B '0' '1' '2' '3'
Data variables:
    CTYPE1          (detector, observation_id, dither) object 512B 'RA---TPV'...
    CTYPE2          (detector, observation_id, dither) object 512B 'DEC--TPV'...
    CD1_1           (detector, observation_id, dither) float64 512B -4.317e-0...
    CD1_2           (detector, observation_id, dither) float64 512B -7.021e-0...
    CD2_1           (detector, observation_id, dither) float64 512B 7.021e-05...
    CD2_2           (detector, observation_id, dither) float64 512B -4.317e-0...
    CRPIX1          (detector, observation_id, dither) float64 512B 4.17e+03 ...
    CRPIX2          (detector, observation_id, dither) float64 512B 4.621e+03...
    CRVAL1          (detector, observation_id, dither) float64 512B 268.5 ......
    CRVAL2          (detector, observation_id, dither) float64 512B 68.23 ......
    NAXIS1          (detector, observation_id, dither) int64 512B 2040 ... 2040
    NAXIS2          (detector, observation_id, dither) int64 512B 2040 ... 2040

In [ ]:
WCS(wcs.sel(detector="DET11", dither="2", observation_id=2683).to_pandas())

WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN' 
CRVAL : 268.6563503943 68.24617668136 
CRPIX : 4170.390230645 4621.116938306 
CD1_1 CD1_2  : -4.297985528424e-05 -7.032797254342e-05 
CD2_1 CD2_2  : 7.03307850190499e-05 -4.29881884951e-05 
NAXIS : 2040  2040

In [ ]:
ds = open_zarr_ref_as_dataset(ref)

In [ ]:
ds

<xarray.Dataset> Size: 10GB
Dimensions:         (observation_id: 1, dither: 4, filter: 3, detector: 16,
                     y: 2040, x: 2040)
Coordinates:
  * detector        (detector) object 128B 'DET11' 'DET12' ... 'DET43' 'DET44'
  * dither          (dither) object 32B '0' '1' '2' '3'
  * filter          (filter) object 24B 'H' 'J' 'Y'
  * observation_id  (observation_id) int64 8B 2683
Dimensions without coordinates: y, x
Data variables:
    DQ              (observation_id, dither, filter, detector, y, x) int32 3GB dask.array<chunksize=(1, 2, 2, 2, 2040, 2040), meta=np.ndarray>
    RMS             (observation_id, dither, filter, detector, y, x) float32 3GB dask.array<chunksize=(1, 2, 2, 2, 2040, 2040), meta=np.ndarray>
    SCI             (observation_id, dither, filter, detector, y, x) float32 3GB dask.array<chunksize=(1, 2, 2, 2, 2040, 2040), meta=np.ndarray>

In [ ]:
median = ds["SCI"].median(dim=["observation_id", "dither", "filter"])

In [ ]:
median.compute()

<xarray.DataArray 'SCI' (detector: 16, y: 2040, x: 2040)> Size: 266MB
array([[[ 64.19491  ,  67.078766 ,  66.44539  , ...,  64.04256  ,
          62.787518 ,  59.350792 ],
        [ 64.54579  ,  63.714226 ,  60.695328 , ...,  62.01454  ,
          63.63827  ,  60.694565 ],
        [ 60.816174 ,  61.2768   ,  63.714226 , ...,  66.12628  ,
          55.953312 ,  55.2466   ],
        ...,
        [ 63.453663 ,  67.74246  ,  62.702446 , ...,  59.646862 ,
          62.916855 ,  63.85218  ],
        [ 64.9606   ,  59.288826 ,  63.397884 , ...,  63.160946 ,
          63.157394 ,  65.6652   ],
        [ 59.76058  ,  60.67249  ,  68.97561  , ...,  62.60383  ,
          58.301983 ,  62.16313  ]],

       [[ 70.7383   ,  69.91258  ,  72.29007  , ...,  -8.780915 ,
          -5.9143744,  -6.920571 ],
        [ 69.49198  ,  67.1319   ,  75.94953  , ...,  -4.9328685,
         -12.070469 , -10.73223  ],
        [ 71.27069  ,  71.91942  ,  63.27051  , ...,  -4.968932 ,
          -4.938717 ,  -5.9631076],
...
         -10.109669 ,  -8.403699 ],
        [ 66.27293  ,  69.674515 ,  61.857292 , ...,  -9.210991 ,
         -16.680946 , -10.271188 ],
        [ 69.171005 ,  69.53067  ,  62.534737 , ..., -17.500885 ,
         -15.873654 ,  -7.4193735]],

       [[ 76.75749  ,  83.60602  ,  83.27513  , ...,  67.20813  ,
          62.931004 ,  68.58031  ],
        [ 74.80112  ,  72.93109  ,  80.6994   , ...,  64.274536 ,
          61.619118 ,  65.80408  ],
        [ 76.75749  ,  74.95543  ,  78.44336  , ...,  71.57124  ,
          72.47168  ,  70.66237  ],
        ...,
        [ 68.75368  ,  69.03995  ,  65.620316 , ...,  68.860085 ,
          64.969086 ,  63.070034 ],
        [ 66.44855  ,  64.932816 ,  72.47544  , ...,  64.05542  ,
          59.470894 ,  71.957886 ],
        [ 68.491776 ,  64.79114  ,  68.05373  , ...,  68.38541  ,
          65.33897  ,  66.87998  ]]],
      shape=(16, 2040, 2040), dtype=float32)
Coordinates:
  * detector  (detector) object 128B 'DET11' 'DET12' 'DET13' ... 'DET43' 'DET44'
Dimensions without coordinates: y, x

Also demonstrate on VIS images...

In [ ]:
fns = sorted(list(path.glob(f"**/2683/EUC_VIS_SWL-DET-*-*-*_*.fits")))

In [ ]:
ref, wcs, zp = create_zarr_ref_from_fits(fns, progress=True)

EUC_VIS_SWL-DET-002683-03-1-0000000__20241017T024937.344633Z.fits, 2683, 3-1, VIS: 100%|██████████| 6/6 [00:43<00:00,  7.26s/it]


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()